In [1]:
def multichannel(filter_func):
    def wrapper(image, *args, **kwargs):
        first_pixel = image[0][0]
        if not isinstance(first_pixel, list):
            return filter_func(image, *args, **kwargs)

        M = len(image)
        N = len(image[0])
        C = len(image[0][0])

        channels = []
        for c in range(C):
            channel = [[image[i][j][c] for j in range(N)] for i in range(M)]
            channels.append(channel)

        filtered = [filter_func(ch, *args, **kwargs) for ch in channels]

        return [[[filtered[c][i][j] for c in range(C)]
                 for j in range(N)] for i in range(M)]

    return wrapper


@multichannel
def svertka(mat, kernel):
    kernel = [row[::-1] for row in kernel[::-1]]

    h, w = len(mat), len(mat[0])
    kh, kw = len(kernel), len(kernel[0])

    ph, pw = kh // 2, kw // 2

    result = [[0.0] * w for _ in range(h)]

    for i in range(h):
        for j in range(w):
            s = 0.0
            for ki in range(kh):
                for kj in range(kw):
                    y = i + ki - ph
                    x = j + kj - pw
                    if 0 <= y < h and 0 <= x < w:
                        s += mat[y][x] * kernel[ki][kj]
            result[i][j] = s

    return result

In [2]:
kernel = [
    [1/9, 1/9, 1/9],
    [1/9, 1/9, 1/9],
    [1/9, 1/9, 1/9],
]

gray = [
    [10, 20, 30, 40],
    [50, 60, 70, 80],
    [90, 100, 110, 120],
    [130, 140, 150, 160],
]
print(svertka(gray, kernel))   

rgb = [
    [[10, 20, 30], [40, 50, 60]],
    [[70, 80, 90], [100, 110, 120]],
]
print(svertka(rgb, kernel))  

[[15.555555555555555, 26.666666666666664, 33.33333333333333, 24.444444444444443], [36.66666666666667, 59.99999999999999, 70.0, 50.0], [63.33333333333333, 100.0, 110.0, 76.66666666666666], [51.111111111111114, 80.0, 86.66666666666666, 59.99999999999999]]
[[[24.444444444444443, 28.88888888888889, 33.33333333333333], [24.444444444444443, 28.88888888888889, 33.33333333333333]], [[24.444444444444443, 28.88888888888889, 33.33333333333333], [24.444444444444443, 28.88888888888889, 33.33333333333333]]]


In [3]:
def rgb_to_yiq(rgb):
    R, G, B = rgb
    Y = 0.299  * R + 0.587  * G + 0.114  * B
    I = 0.5959 * R - 0.2746 * G - 0.3213 * B
    Q = 0.2115 * R - 0.5227 * G + 0.3112 * B
    return [Y, I, Q]


def yiq_to_rgb(yiq):
    Y, I, Q = yiq
    R = Y + 0.956 * I + 0.619 * Q
    G = Y - 0.272 * I - 0.647 * Q
    B = Y - 1.106 * I + 1.703 * Q
    return [R, G, B]


def convert(color):

    c1, c2, c3, t = color

    if t == 0:
        Y, I, Q = rgb_to_yiq([c1, c2, c3])
        return [Y, I, Q, 1]
    elif t == 1:
        R, G, B = yiq_to_rgb([c1, c2, c3])
        return [R, G, B, 0]

In [4]:
print(convert([255, 0, 0, 0]))

print(convert([76.245, 151.9545, 53.9325, 1]))

print(convert([0, 0, 0, 0]))  
print(convert([0, 0, 0, 1]))  

print(convert([255, 255, 255, 0]))
print(convert([255, 0, 0, 1]))   

print(convert([0, 255, 0, 0]))  

print(convert([0, 0, 255, 0])) 

[76.24499999999999, 151.9545, 53.9325, 1]
[254.8977195, 0.019048499999996693, 0.03037050000000363, 0]
[0.0, 0.0, 0.0, 1]
[0.0, 0.0, 0.0, 0]
[255.0, 0.0, -2.842170943040401e-14, 1]
[255.0, 255.0, 255.0, 0]
[149.685, -70.023, -133.28850000000003, 1]
[29.07, -81.9315, 79.356, 1]
